# Ag402 Quick Start: AI Agent Auto-Pay in 5 Minutes

**Give AI Agents the ability to autonomously pay for API calls** — HTTP 402 standard · Solana USDC settlement · Two lines of code

> **For**: AI Agent developers who want their agents to access paid APIs automatically.
>
> **适用人群**: 希望 Agent 能自动访问付费 API 的 AI 开发者。

This notebook walks you through the full [x402 payment protocol](https://github.com/coinbase/x402) — **zero local setup, runs entirely in your browser**.

### What You'll Do / 你将体验

| Step | What Happens | The Point |
|------|-------------|----------|
| **Step 1** | Start a paid weather API | See how *any* API becomes a paid service |
| **Step 2** | Call it without paying | See the HTTP 402 rejection |
| **Step 3** | Let Ag402 auto-pay | See 402 → pay → 200 in one call |
| **Step 4** | Check wallet & history | See where the money went |
| **Step 5** | Try it yourself | Change cities, see balance drop |
| **Step 6** | Two-line integration | See how simple it is in YOUR code |

> This demo uses **test mode** (mock Solana) — no real funds or blockchain needed.
>
> 本 Demo 使用测试模式（Mock Solana），无需真实资金或区块链。

In [ ]:
# Install dependencies / 安装依赖
!pip install -q ag402-core ag402-mcp nest_asyncio

## Step 1: Start a Paid API / 启动一个付费 API

In the x402 model, any HTTP API becomes a **paid service** by placing an Ag402 gateway in front of it:

```
                   ┌────────────────┐     ┌──────────────┐
  Client  ──▶  │ x402 Gateway   │ ──▶ │  Your API     │
                   │ (checks payment)│     │ (weather etc.)│
                   └────────────────┘     └──────────────┘
              No payment? → 402        Has proof? → 200 + data
```

Below we start a mock weather API + x402 gateway inside this notebook. The boilerplate is collapsed — the important part is the **gateway configuration** at the bottom (price, chain, token, address).

> 下面我们在 Notebook 内启动服务。样板代码已折叠，重点看底部的网关配置。

In [ ]:
# --- Boilerplate: mock weather API + background server helper ---
# --- 样板代码：Mock 天气 API + 后台服务 helper（可忽略细节） ---

import asyncio, random, uvicorn, nest_asyncio
nest_asyncio.apply()  # Colab already has a running event loop

from fastapi import FastAPI
from fastapi.responses import JSONResponse

weather_app = FastAPI(title="Mock Weather Service")
WEATHER_DATA = {
    "tokyo": {"city": "Tokyo", "temp": 22, "condition": "sunny", "humidity": "45%"},
    "london": {"city": "London", "temp": 14, "condition": "cloudy", "humidity": "78%"},
    "new york": {"city": "New York", "temp": 18, "condition": "partly cloudy", "humidity": "55%"},
    "beijing": {"city": "Beijing", "temp": 20, "condition": "hazy", "humidity": "60%"},
    "san francisco": {"city": "San Francisco", "temp": 17, "condition": "foggy", "humidity": "82%"},
}

@weather_app.get("/weather")
async def get_weather(city: str = "Tokyo"):
    key = city.lower().strip()
    if key in WEATHER_DATA:
        return JSONResponse(content=WEATHER_DATA[key])
    return JSONResponse(content={"city": city, "temp": random.randint(5, 35),
                                 "condition": random.choice(["sunny","cloudy","rainy"]), "humidity": f"{random.randint(30,90)}%"})

class BackgroundServer:
    def __init__(self, app, host="127.0.0.1", port=0):
        self.app, self.host, self.port = app, host, port
        self._server = self._task = None
    async def start(self):
        config = uvicorn.Config(self.app, host=self.host, port=self.port, log_level="error", lifespan="off")
        self._server = uvicorn.Server(config)
        self._server.install_signal_handlers = lambda: None
        self._task = asyncio.create_task(self._server.serve())
        for _ in range(50):
            await asyncio.sleep(0.1)
            if self._server.started: break
    async def stop(self):
        if self._server: self._server.should_exit = True
        if self._task:
            try: await asyncio.wait_for(self._task, timeout=3.0)
            except (asyncio.TimeoutError, asyncio.CancelledError): self._task.cancel()

print("Boilerplate ready.")

In [ ]:
# --- THE IMPORTANT PART: configure and start the x402 payment gateway ---
# --- 重点：配置并启动 x402 支付网关 ---

from ag402_core.gateway.auth import PaymentVerifier
from ag402_mcp.gateway import X402Gateway

WEATHER_PORT = 18000
GATEWAY_PORT = 18001
RECIPIENT = "DemoRecipientWa11et11111111111111111111"

async def _start_servers():
    global weather_server, gateway_server

    weather_server = BackgroundServer(weather_app, port=WEATHER_PORT)
    await weather_server.start()

    gateway = X402Gateway(
        target_url=f"http://127.0.0.1:{WEATHER_PORT}",
        price="0.02",       # $0.02 USDC per request
        chain="solana",     # Settlement chain
        token="USDC",       # Payment token
        address=RECIPIENT,  # Seller's wallet
        verifier=PaymentVerifier(),  # Test mode
    )
    gateway_server = BackgroundServer(gateway.create_app(), port=GATEWAY_PORT)
    await gateway_server.start()

asyncio.get_event_loop().run_until_complete(_start_servers())

print(f"\n{'='*60}")
print(f"  Weather API : http://127.0.0.1:{WEATHER_PORT}/weather")
print(f"  x402 Gateway: http://127.0.0.1:{GATEWAY_PORT}/weather")
print(f"  Price       : 0.02 USDC per request")
print(f"  Chain       : Solana | Token: USDC")
print(f"{'='*60}")
print(f"\nGateway is live. Any request without payment proof \u2192 HTTP 402.")

## Step 2: Call WITHOUT Payment — See the 402 Wall / 不付费，被拒

What happens when you call the gateway **without paying**? Let's find out.

> 不带支付证明直接请求网关，看看会怎样。

In [ ]:
import httpx, json

url = f"http://127.0.0.1:{GATEWAY_PORT}/weather?city=Tokyo"
resp = httpx.get(url)

# --- Visual output ---
print(f"\n  \u2718 Status: {resp.status_code} Payment Required")
print(f"  \u2502")

if resp.status_code == 402:
    try:
        body = resp.json()
        print(f"  \u2502 The gateway says: \"Pay me first.\"")
        print(f"  \u2502")
        print(f"  \u2502 Payment challenge:")
        for k, v in body.items():
            print(f"  \u2502   {k}: {v}")
    except Exception:
        print(f"  \u2502 Raw response: {resp.text[:300]}")

    www_auth = resp.headers.get("www-authenticate", "")
    if www_auth:
        print(f"  \u2502")
        print(f"  \u2502 WWW-Authenticate: {www_auth[:120]}...")

print(f"  \u2502")
print(f"  \u2514\u2500\u2500 We got blocked! Need to pay 0.02 USDC to get the weather data.")
print(f"")
print(f"  Next: let Ag402 handle this automatically \u2192")

## Step 3: Auto-Pay with Ag402 — The Magic Moment / 自动支付

Now watch. The **same URL**, but this time Ag402 middleware sits in front:

```
Your code calls GET /weather?city=Tokyo
         \u2502
         \u25bc
    Ag402 sends request \u2192 gets 402 \u2192 parses challenge
    \u2192 checks budget ($0.02 < $5 limit? \u2713)
    \u2192 pays on-chain \u2192 retries with proof
         \u2502
         \u25bc
    You get 200 + weather data. You saw nothing.
```

> 你的代码只看到 200 和数据。中间的 402 → 支付 → 重试全部透明发生。

In [ ]:
import time
from ag402_core.wallet.agent_wallet import AgentWallet
from ag402_core.payment.solana_adapter import MockSolanaAdapter
from ag402_core.middleware.x402_middleware import X402PaymentMiddleware
from ag402_core.config import X402Config, RunMode

async def demo_auto_pay():
    global wallet, middleware

    # Initialize wallet with $100 test funds
    # 初始化钱包，充入 $100 测试资金
    wallet = AgentWallet(db_path=":memory:")
    await wallet.init_db()
    await wallet.deposit(100.00, note="Test faucet")
    balance_before = await wallet.get_balance()

    # Mock Solana adapter (no real blockchain)
    provider = MockSolanaAdapter(balance=1000.0, address="Ag402DemoAddr111111111111111111111111111111")
    config = X402Config(mode=RunMode.TEST, single_tx_limit=1.0)
    middleware = X402PaymentMiddleware(wallet=wallet, provider=provider, config=config)

    # === THE SAME URL AS STEP 2 ===
    url = f"http://127.0.0.1:{GATEWAY_PORT}/weather?city=Tokyo"

    print(f"  Wallet: ${balance_before:.2f} USDC")
    print(f"  Sending: GET {url}")
    print(f"  ...\n")

    t0 = time.monotonic()
    result = await middleware.handle_request("GET", url)
    elapsed = time.monotonic() - t0
    balance_after = await wallet.get_balance()

    # --- Before / After comparison ---
    print(f"  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 BEFORE (Step 2) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510")
    print(f"  \u2502  httpx.get(url) \u2192 402 Payment Required    \u2718  \u2502")
    print(f"  \u2502  No data. Blocked.                           \u2502")
    print(f"  \u251c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 AFTER  (Step 3) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524")

    if result.status_code == 200:
        data = json.loads(result.body)
        print(f"  \u2502  middleware.handle_request(url) \u2192 200 OK    \u2714  \u2502")
        print(f"  \u2502                                                \u2502")
        print(f"  \u2502  City:      {data['city']:<20}               \u2502")
        print(f"  \u2502  Temp:      {data['temp']}\u00b0C                              \u2502")
        print(f"  \u2502  Condition: {data['condition']:<20}               \u2502")
        if 'humidity' in data:
            print(f"  \u2502  Humidity:  {data['humidity']:<20}               \u2502")
    else:
        print(f"  \u2502  Status: {result.status_code} \u2014 {result.error}")

    print(f"  \u251c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 Payment Details \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524")
    if result.payment_made:
        print(f"  \u2502  Amount:   ${result.amount_paid:.2f} USDC                       \u2502")
        print(f"  \u2502  TX Hash:  {result.tx_hash[:32]}...     \u2502")
        print(f"  \u2502  Time:     {elapsed:.2f}s                                \u2502")
    print(f"  \u2502                                                \u2502")
    print(f"  \u2502  Balance:  ${balance_before:.2f} \u2192 ${balance_after:.2f} USDC (-${float(balance_before - balance_after):.2f})  \u2502")
    print(f"  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518")

asyncio.get_event_loop().run_until_complete(demo_auto_pay())

## Step 4: Wallet & Transaction History / 钱包与交易记录

Every payment is tracked. Let's inspect the wallet.

In [ ]:
async def show_wallet():
    balance = await wallet.get_balance()
    stats = await wallet.get_summary_stats()
    transactions = await wallet.get_transactions(limit=10)

    print(f"  \u250c\u2500 Wallet Dashboard \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510")
    print(f"  \u2502  Balance:       ${balance:.2f} USDC")
    print(f"  \u2502  Transactions:  {stats.get('tx_count', 0)}")
    print(f"  \u2502  Today's Spend: ${float(stats.get('today_spend', 0)):.2f}")
    print(f"  \u2502  Total Spend:   ${float(stats.get('total_spend', 0)):.2f}")
    print(f"  \u251c\u2500 Transaction History \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524")
    print(f"  \u2502  {'Type':<12} {'Amount':>10}  {'Detail':<28}")
    print(f"  \u2502  {'\u2500'*12} {'\u2500'*10}  {'\u2500'*28}")
    for tx in transactions:
        amount_str = f"${float(tx.amount):>8.2f}"
        detail = tx.note if tx.note else (tx.tx_hash[:24] + "..." if tx.tx_hash else "\u2014")
        print(f"  \u2502  {tx.type:<12} {amount_str:>10}  {detail:<28}")
    print(f"  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518")

asyncio.get_event_loop().run_until_complete(show_wallet())

## Step 5: Try It Yourself! / 自己试试！

Change the city, run multiple requests, watch your balance go down. Available cities: `Tokyo`, `London`, `New York`, `Beijing`, `San Francisco` — or type any city name.

> 改改城市名，多跑几次，看余额变化。

In [ ]:
# Change the city here and re-run this cell!
# 修改城市名然后重新运行这个 Cell！
CITY = "London"  # <-- TRY: "Tokyo", "Beijing", "Paris", "Mumbai", anything!

async def try_it():
    balance_before = await wallet.get_balance()
    url = f"http://127.0.0.1:{GATEWAY_PORT}/weather?city={CITY}"
    result = await middleware.handle_request("GET", url)
    balance_after = await wallet.get_balance()

    if result.status_code == 200:
        data = json.loads(result.body)
        print(f"  {data['city']}: {data['temp']}\u00b0C, {data['condition']}")
        if 'humidity' in data:
            print(f"  Humidity: {data['humidity']}")
    else:
        print(f"  Error: {result.status_code} \u2014 {result.error}")

    print(f"  Paid: ${result.amount_paid:.2f} USDC | Balance: ${balance_after:.2f} (was ${balance_before:.2f})")

    # Show running total
    txs = await wallet.get_transactions(limit=50)
    payments = [tx for tx in txs if tx.type == "deduction"]
    print(f"  Total requests so far: {len(payments)} | Total spent: ${sum(float(tx.amount) for tx in payments):.2f} USDC")

asyncio.get_event_loop().run_until_complete(try_it())

## What Just Happened? / 刚才发生了什么？

```
Your AI Agent              Ag402 Middleware           Target API
     \u2502                             \u2502                          \u2502
     \u2502\u2500\u2500 GET /weather \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u25b6\u2502                          \u2502
     \u2502                             \u2502\u2500\u2500 GET /weather \u2500\u2500\u2500\u2500\u2500\u2500\u25b6\u2502
     \u2502                             \u2502\u25c0\u2500\u2500 402 + challenge \u2500\u2500\u2500\u2500\u2500\u2502
     \u2502                             \u2502                          \u2502
     \u2502                             \u2502  \u2460 Parse payment terms   \u2502
     \u2502                             \u2502  \u2461 Check budget \u2713        \u2502
     \u2502                             \u2502  \u2462 Solana on-chain tx    \u2502
     \u2502                             \u2502  \u2463 Retry with proof      \u2502
     \u2502                             \u2502                          \u2502
     \u2502                             \u2502\u2500\u2500 GET + proof \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u25b6\u2502
     \u2502                             \u2502\u25c0\u2500\u2500 200 OK + data \u2500\u2500\u2500\u2500\u2500\u2500\u2502
     \u2502\u25c0\u2500\u2500 200 OK + data \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2502                          \u2502
```

**Steps \u2460\u2461\u2462\u2463 are invisible to your code.** You always get a normal 200.

### Built-in Safety / \u5185\u7f6e\u5b89\u5168\u673a\u5236

Even if your agent loops infinitely, it **cannot drain the wallet**:

| Layer | Default | What it does |
|-------|---------|-------------|
| Single-TX cap | **$5.00** | Rejects any request over $5 |
| Per-minute | **$2.00 / 5 txns** | Rate limit |
| Daily cap | **$10.00** | Hard ceiling $1,000 |
| Circuit breaker | **3 failures** | Auto-stops, 60s cooldown |
| Auto-rollback | Always on | Failed payment \u2192 money returned |

## Step 6: Two Lines in YOUR Code / 两行代码集成到你的项目

Everything above was the *educational breakdown*. In real life, you don't touch middleware or wallets. The entire integration is **two lines**:

In [ ]:
# This is ALL you need in production. Two lines. That's it.
# 这就是生产环境的全部代码。两行。

import ag402_core

# ag402_core.enable() patches httpx and requests globally.
# Every HTTP 402 response is auto-handled: parse challenge -> pay -> retry.
# Your existing code needs ZERO changes.
#
# In this notebook we already have middleware set up, so we'll show the
# context-manager form instead (scoped, doesn't conflict with our demo):

print("In your project, it's just:")
print()
print("    import ag402_core")
print("    ag402_core.enable()")
print()
print("    # Then your existing code works as-is:")
print("    response = httpx.get('https://paid-api.example.com/data')")
print("    # 402? Ag402 auto-pays. You always get 200.")
print()
print("Or use the CLI:")
print("    ag402 run -- python my_agent.py")
print()
print("Works with LangChain, AutoGen, CrewAI, Cursor, Claude Code \u2014")
print("anything that uses httpx or requests under the hood.")

## Real-World Example / 真实案例

### Token RugCheck — Solana Token Safety Audit

[**token-bugcheck**](https://github.com/AetherCore-Dev/token-bugcheck) is a production service where AI Agents pay **0.05 USDC per audit** to detect rug pulls before buying tokens. The entire integration:

```python
# seller (audit API provider) — add paywall to existing API:
ag402 serve --target http://localhost:8000 --price 0.05

# buyer (AI agent) — two lines, zero changes to business logic:
import ag402_core
ag402_core.enable()
result = httpx.get("https://audit-api.example.com/check?token=So11...")
```

> **Any HTTP API becomes a paid service. Any AI agent becomes a paying customer. Zero changes to business logic on either side.**

---

### Next Steps / 下一步

| What | How |
|------|-----|
| Install locally | `pip install ag402-core && ag402 setup && ag402 demo` |
| Real on-chain (local) | [Localnet Guide](https://github.com/AetherCore-Dev/ag402/blob/main/docs/guide-localnet.md) |
| Add to Cursor/Claude Code | `ag402 install cursor` or `ag402 install claude-code` |
| Build your own paid API | `ag402 serve --target http://localhost:8000 --price 0.10` |
| Full docs | [github.com/AetherCore-Dev/ag402](https://github.com/AetherCore-Dev/ag402) |

If this was useful, [**star us on GitHub**](https://github.com/AetherCore-Dev/ag402)!

In [ ]:
# Cleanup / 清理

async def cleanup():
    errors = []
    for name, coro in [("middleware", lambda: middleware.close()),
                       ("gateway",    lambda: gateway_server.stop()),
                       ("weather",    lambda: weather_server.stop())]:
        try:
            await coro()
        except Exception as e:
            errors.append(f"{name}: {e}")
    if errors:
        print(f"  Cleanup warnings: {'; '.join(errors)}")
    print("  All servers stopped.")

asyncio.get_event_loop().run_until_complete(cleanup())
print("  Done! https://github.com/AetherCore-Dev/ag402")